In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

In [ ]:
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")


In [ ]:
num_df = train_df.select_dtypes(include=[np.number])
cat_df = train_df.select_dtypes(include=['object'])

In [ ]:
corr = num_df.corr()
num_cols = corr[(corr['SalePrice'] > 0.3) | (corr['SalePrice'] < -0.3)].index
num_df = num_df[num_cols]

In [ ]:
for col in ["LotFrontage", "GarageYrBlt", "MasVnrArea"]:
    num_df[col] = num_df[col].fillna(num_df[col].median())


In [ ]:
# ---------------------------
# Fill missing categorical values
# ---------------------------

# Fill some columns with a placeholder string
cat_df.fillna({
    'PoolQC': 'No Value', 
    'MiscFeature': 'No Value', 
    'Alley': 'No Value', 
    'Fence': 'No Value', 
    'FireplaceQu': 'No Value'
}, inplace=True)

# Fill remaining columns with the mode (most frequent value)
for col in ['GarageCond','GarageQual','GarageFinish','GarageType',
            'BsmtFinType2','BsmtExposure','BsmtFinType1','BsmtQual',
            'BsmtCond','MasVnrType','Electrical']:
    cat_df[col] = cat_df[col].fillna(cat_df[col].mode()[0])

In [ ]:
df = pd.concat([num_df, cat_df], axis=1)
X = pd.get_dummies(df.drop('SalePrice', axis=1))
y = df['SalePrice']

In [ ]:
X = X.astype(np.float32)

y = y.astype(np.float32)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.1, random_state=73
)

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)


In [ ]:
n_input = X_train.shape[1]
net = nn.Sequential(
    nn.Linear(n_input, n_input*2),
    nn.ELU(),
    nn.Linear(n_input*2, n_input//2),
    nn.Tanh(),
    nn.Linear(n_input//2, n_input//4),
    nn.ELU(),
    nn.Linear(n_input//4, 1)  # Regression output
)
print(net)



In [ ]:
loss_func = nn.MSELoss()  # Change to MSE for regression
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)


In [ ]:
loss_history = []

for i in range(200):
    optimizer.zero_grad()
    y_pred = net(X_train)
    loss = loss_func(y_pred, y_train)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(loss_history)
plt.title('Training Loss')
plt.ylabel('MSE Loss')
plt.xlabel('Iteration')
plt.show()


In [ ]:
from sklearn.metrics import r2_score

y_pred_test = net(X_test).detach().numpy()
print("R2 Score:", r2_score(y_test.numpy(), y_pred_test))

Using Tensorflow

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam


In [ ]:
X_train_np = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
X_test_np = X_test.values if isinstance(X_test, pd.DataFrame) else X_test
y_train_np = y_train.values.reshape(-1,1) if isinstance(y_train, pd.Series) else y_train.reshape(-1,1)
y_test_np = y_test.values.reshape(-1,1) if isinstance(y_test, pd.Series) else y_test.reshape(-1,1)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input


In [ ]:
model = Sequential([
    Input(shape=(n_input,)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='linear')
])


In [ ]:
model.compile(optimizer=Adam(learning_rate=0.01), loss='mse', metrics=['mae'])


In [ ]:
history = model.fit(
    X_train_np, y_train_np,
    validation_split=0.1,
    epochs=200,
    batch_size=32,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import r2_score

y_pred = model.predict(X_test_np)
print("R2 Score:", r2_score(y_test_np, y_pred))